# 04c — Fine-tune Indic-Transcribe-Flex (mixed-script mode)

A full fine-tune of `bodhan-ai/indic-transcribe-flex`: a 1.2 B Canary-style encoder-decoder with
a 32-layer conformer encoder and a 24-layer transformer decoder. It trains on the 18.3 h train
split with the `ne` + mixed-script prompt, uses val for model selection and gold for the final
score.

**Why this model.** It led the zero-shot comparisons at 18.2% folded WER on gold (CI 14.0–21.2),
flat across code-mixing tiers. Its mixed mode already writes the corpus's convention: 33.6% Latin
tokens against the references' 36.0%.

**What had to be built, because the release is inference-only.**
- **Loss.** The model's `forward(labels=...)` raises `NotImplementedError`. The loss here is
  cross-entropy on teacher-forced logits. The input is the fixed 10-token prompt followed by the
  target, and loss is taken on the target tokens and the end-of-text only.
- **Target encoding.** The tokenizer has no text `encode()`. Targets are the multilingual
  SentencePiece ids offset by the 1,152 special tokens, which round-trips exactly under the
  model's own `decode`.
- **Regularisation.** The port has no dropout, so SpecAugment (2 frequency masks of ≤27 bins and
  10 time masks of ≤5%, Canary's defaults) is applied to the features on the GPU.
- **Labels mapped to what the tokenizer can write.** Without this, 76% of labels would contain an
  unknown token. `।` becomes `.`, Devanagari digits become Latin, and ZWJ/ZWNJ are removed.
  `fold.py` scores each pair as identical, so WER is unaffected.

**Licence.** A fine-tuned model is a derivative under the Indic Open Model License v1.0. You can
use it privately. Giving it to anyone passes the same licence on, and hosting it as a service for
others needs Bodhan AI's written sign-off. Keep the weights private.

**Choices.** Peak LR 1e-5. Linear decay, 10% warmup, up to 6 epochs.

**Decoding: greedy, plus a loop retry.** A clip whose greedy output repeats a 3-word sequence 5+
times is decoded again, alone, with a repetition penalty, no repeated 6-token phrase and a
length cap from its duration. The trigger reads only the model's own output (the idea of
Whisper's compression-ratio fallback), so it is usable on any audio. The settings were fixed
before scoring and never tuned; the Gold cell records the greedy-only score from the same run.
Run 2026-09-12: gold 13.20% greedy -> 11.44% with the retry (7 loops -> 0); val unchanged at 7.60%.

**Keeping the A100 busy** (all in `ftkit.py`):
- **No disk I/O in the loop.** Audio sits in RAM as int16, and a clip is a slice of its episode.
- **Little padding.** Batches are built by duration and padded to whole seconds, so there are only
  ~20 distinct shapes and cudnn picks its kernels once per shape.
- **The CPU never stalls the GPU.** DataLoader workers collate and pin the next batches while the
  current one runs.
- **Measured batch size.** A probe finds the largest micro-batch that survives forward+backward on
  the longest clip, with the optimizer state already allocated. Training uses 90% of it, and
  gradient accumulation makes up the effective batch.
- **A100 arithmetic.** bf16 autocast, TF32 matmuls and fused AdamW.
- **Measured, not assumed.** The log reports throughput (× realtime), padding waste, GPU
  utilisation and memory every few steps. Single-digit utilisation or high padding waste means a
  setting needs changing.

## Config

In [ ]:
RUN_NAME = "indic-transcribe-flex-ft"
MODEL_ID = "bodhan-ai/indic-transcribe-flex"
LANG, MODE = "ne", "mixed"
USE_DRIVE = True
EPOCHS, LR, WARMUP = 6, 1e-5, 0.1
# Shown on the harness's Models page (D83). Say what is different about this run.
MODEL_NAME = "Indic-Transcribe-Flex FT"
MODEL_DESCRIPTION = "04c full fine-tune, standard settings."
EFFECTIVE_S = 720.0       # ~12 min of audio per optimizer step
PAD_TO_S = 1.0
PROBE_FRACTION = 0.9
GRAD_CKPT = False         # checkpoint the conformer layers if the probe finds a small batch
EVAL_BUDGET_S, EVAL_ITEMS = 1200.0, 96

## Setup

In [ ]:
%pip install -q rapidfuzz
import json
import os
import sys
from pathlib import Path

# Before torch touches CUDA: lets the allocator grow segments instead of fragmenting when batch
# shapes vary (a fresh kernel only; the Omnilingual subprocess inherits it).
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

IN_COLAB = "google.colab" in sys.modules
# Secrets and the Drive consent popup only work from a cell run in the Colab UI. When cells are
# driven from outside (the Colab MCP), run this cell by hand once. The token is then kept in the
# hub's own token file on the VM (where `hf auth login` puts it), so a kernel restart needs no
# click; the file goes when the runtime is deleted.
if IN_COLAB:
    from google.colab import userdata

    TOKEN_FILE = Path.home() / ".cache" / "huggingface" / "token"
    if not os.environ.get("HF_TOKEN"):
        os.environ["HF_TOKEN"] = (TOKEN_FILE.read_text().strip() if TOKEN_FILE.exists()
                                  else userdata.get("HF_TOKEN"))
    if not TOKEN_FILE.exists():
        TOKEN_FILE.parent.mkdir(parents=True, exist_ok=True)
        TOKEN_FILE.write_text(os.environ["HF_TOKEN"])
        TOKEN_FILE.chmod(0o600)
    if USE_DRIVE and not Path("/content/drive/MyDrive").exists():
        from google.colab import drive

        drive.mount("/content/drive")
FT = Path("/content/ft") if IN_COLAB else Path.cwd() / ".cache-ft"
FT.mkdir(parents=True, exist_ok=True)
OUT = (Path("/content/drive/MyDrive/nepanglish-asr") if IN_COLAB and USE_DRIVE else FT / "out") / RUN_NAME
OUT.mkdir(parents=True, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print("run:", RUN_NAME, "| outputs:", OUT)

In [ ]:
%%writefile /content/ft/ftkit.py
"""Shared fine-tuning kit for the Nepanglish ASR notebook (04c).

Everything that is not model-specific: the dataset held in RAM, duration-bucketed batches, the
harness scorer, a GPU utilisation monitor, a batch-size probe and one training loop. The notebook
writes this file out, so it imports exactly the same code as the main kernel. Keep it importable
on Python 3.10+ with numpy 1.x or 2.x.

How the GPU is kept busy:
  * audio is decoded once into RAM as int16; a clip is a slice, so no disk I/O in the loop;
  * batches are built by duration and padded to whole seconds, so there is little padding and
    only ~20 distinct shapes (cudnn.benchmark can then pick kernels once per shape);
  * DataLoader workers collate and pin the next batches while the GPU runs the current one;
  * a probe finds the largest micro-batch that survives forward+backward at the longest clip,
    with the optimizer state already allocated, and training uses a fixed fraction of it;
  * bf16 autocast, TF32 matmuls and fused AdamW; gradient accumulation reaches the effective
    batch; utilisation, throughput and padding waste are logged, not assumed.
"""

from __future__ import annotations

import json
import math
import random
import re
import shutil
import subprocess
import sys
import threading
import time
from collections import Counter
from collections.abc import Callable, Sequence
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import numpy as np
import soundfile as sf
import torch

REPO = "Sagyam/nepanglish-asr"
SR = 16_000


# --- data --------------------------------------------------------------------------------------


def download_dataset(local: str | None = None) -> Path:
    """The HF dataset snapshot, or a local copy in the same layout."""
    if local:
        return Path(local)
    from huggingface_hub import snapshot_download

    return Path(
        snapshot_download(
            REPO, repo_type="dataset", allow_patterns=["training/*", "gold/*", "harness/*"]
        )
    )


def load_splits(data: Path) -> dict[str, list[dict]]:
    """train / val from the training export, gold from the gold export; asserts disjointness."""
    rows = [
        json.loads(line) for line in (data / "training" / "training.jsonl").open(encoding="utf-8")
    ]
    gold = [json.loads(line) for line in (data / "gold" / "gold.jsonl").open(encoding="utf-8")]
    splits = {
        "train": [r for r in rows if r["split"] == "train"],
        "val": [r for r in rows if r["split"] == "val"],
        "gold": gold,
    }
    trained = {r["segment_id"] for r in rows}
    assert not trained & {r["segment_id"] for r in gold}, "a gold clip is in the training export"
    return splits


def duration(row: dict) -> float:
    return row["end_time"] - row["start_time"]


class AudioStore:
    """Every episode decoded once into RAM as int16; a clip is a slice of it (a view, no copy)."""

    def __init__(self, data: Path, episode_ids: Sequence[str]):
        self.audio: dict[str, np.ndarray] = {}
        for ep in sorted(set(episode_ids)):
            audio, sr = sf.read(data / "training" / "episodes" / f"{ep}.flac", dtype="int16")
            assert sr == SR and audio.ndim == 1, (ep, sr, audio.shape)
            self.audio[ep] = audio

    def clip(self, row: dict) -> np.ndarray:
        audio = self.audio[row["episode_id"]]
        return audio[round(row["start_time"] * SR) : round(row["end_time"] * SR)]

    def clip_f32(self, row: dict) -> torch.Tensor:
        return torch.from_numpy(self.clip(row).astype(np.float32) / 32768.0)

    @property
    def gib(self) -> float:
        return sum(a.nbytes for a in self.audio.values()) / 2**30


def bucket_batches(
    rows: Sequence[dict],
    *,
    budget_s: float,
    max_items: int,
    pad_to_s: float,
    shuffle: bool,
    seed: int = 0,
) -> list[list[int]]:
    """Index batches whose padded size (items x longest clip, rounded up to `pad_to_s`) fits
    `budget_s`. Sorting by duration keeps padding low; shuffling reorders whole batches and
    jitters lengths slightly so the same clips do not always share a batch."""
    rng = random.Random(seed)
    jitter = [rng.uniform(-0.3, 0.3) if shuffle else 0.0 for _ in rows]
    order = sorted(range(len(rows)), key=lambda i: duration(rows[i]) + jitter[i])
    batches: list[list[int]] = []
    cur: list[int] = []
    longest = 0.0
    for i in order:
        d = math.ceil(duration(rows[i]) / pad_to_s) * pad_to_s
        if cur and ((len(cur) + 1) * max(longest, d) > budget_s or len(cur) >= max_items):
            batches.append(cur)
            cur, longest = [], 0.0
        cur.append(i)
        longest = max(longest, d)
    if cur:
        batches.append(cur)
    if shuffle:
        rng.shuffle(batches)
    return batches


def pad_len(samples: int, pad_to_s: float) -> int:
    step = int(pad_to_s * SR)
    return math.ceil(samples / step) * step


class _Batches(torch.utils.data.Dataset):
    def __init__(self, rows, batches, collate):
        self.rows, self.batches, self.collate = rows, batches, collate

    def __len__(self):
        return len(self.batches)

    def __getitem__(self, i):
        return self.collate([self.rows[j] for j in self.batches[i]])


def loader(rows, batches, collate, workers: int) -> torch.utils.data.DataLoader:
    return torch.utils.data.DataLoader(
        _Batches(rows, batches, collate),
        batch_size=None,
        shuffle=False,
        num_workers=workers,
        pin_memory=True,
        prefetch_factor=4 if workers else None,
        persistent_workers=False,
    )


# --- scoring -----------------------------------------------------------------------------------


def is_loop(text: str) -> bool:
    """A 3-word sequence repeated 5 or more times: the decoder is stuck, not transcribing."""
    toks = text.split()
    top = Counter(zip(toks, toks[1:], toks[2:], strict=False)).most_common(1)
    return bool(top) and top[0][1] >= 5


class RetryLoops:
    """Wraps a batch `transcribe`: a clip whose first decode loops is decoded again, alone, by
    `retry` (anti-repetition settings); every other clip keeps its first decode untouched.
    `log` keeps (segment_id, first, retried), so the effect is measurable within one run."""

    def __init__(self, transcribe: Callable[[list[dict]], list[str]], retry: Callable[[dict], str]):
        self.transcribe, self.retry = transcribe, retry
        self.log: list[tuple[str, str, str]] = []

    def __call__(self, rows: list[dict]) -> list[str]:
        texts = self.transcribe(rows)
        for i, text in enumerate(texts):
            if is_loop(text):
                texts[i] = self.retry(rows[i])
                self.log.append((rows[i]["segment_id"], text, texts[i]))
        return texts


def harness_scorer(data: Path, work: Path) -> Callable[[Sequence[str], Sequence[str]], dict]:
    """fold.py from the dataset's harness/, imported from a real copy: the HF cache stores files as
    symlinks into its blob store, and normalize.py finds its config via its resolved path."""
    dst = work / "harness"
    if not dst.exists():
        shutil.copytree(data / "harness", dst)
    for name in [m for m in sys.modules if m == "app" or m.startswith("app.")]:
        del sys.modules[name]
    sys.path.insert(0, str(dst / "backend"))
    from app.services.fold import fold_tokens, word_errors
    from rapidfuzz.distance import Levenshtein

    dev = re.compile(r"[ऀ-ॿ]")

    def chars(text: str) -> str:
        return " ".join(t if dev.search(t) else t.lower() for t in fold_tokens(text))

    def score(refs: Sequence[str], hyps: Sequence[str]) -> dict:
        words = werr = rwords = rerr = nchars = cerr = loops = 0
        for ref, hyp in zip(refs, hyps, strict=True):
            words += len(fold_tokens(ref))
            werr += word_errors(ref, hyp).errors
            raw = word_errors(ref, hyp, folded=False)
            rerr, rwords = rerr + raw.errors, rwords + raw.ref_words
            nchars += len(chars(ref))
            cerr += Levenshtein.distance(chars(ref), chars(hyp))
            loops += is_loop(hyp)
        return {
            "wer": 100 * werr / max(words, 1),
            "raw_wer": 100 * rerr / max(rwords, 1),
            "cer": 100 * cerr / max(nchars, 1),
            "loops": loops,
            "clips": len(refs),
        }

    return score


def transcribe_rows(
    rows: Sequence[dict],
    transcribe: Callable[[list[dict]], list[str]],
    *,
    budget_s: float,
    max_items: int,
    pad_to_s: float,
) -> tuple[list[str], list[float]]:
    """Run `transcribe` over duration-bucketed batches; returns texts and per-clip compute seconds
    in the original row order."""
    texts: list[str] = [""] * len(rows)
    compute = [0.0] * len(rows)
    for batch in bucket_batches(
        rows, budget_s=budget_s, max_items=max_items, pad_to_s=pad_to_s, shuffle=False
    ):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        out = transcribe([rows[i] for i in batch])
        torch.cuda.synchronize()
        took = time.perf_counter() - t0
        audio_s = sum(duration(rows[i]) for i in batch)
        for i, text in zip(batch, out, strict=True):
            texts[i] = text.strip()
            compute[i] = took * duration(rows[i]) / audio_s
    return texts, compute


def write_hyps(
    path: Path, rows: Sequence[dict], texts: Sequence[str], compute: Sequence[float]
) -> None:
    """The hypothesis-cache format the bake-off comparisons used, so the result can join them."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as fh:
        for r, text, c in zip(rows, texts, compute, strict=True):
            fh.write(
                json.dumps(
                    {"segment_id": r["segment_id"], "text": text, "compute_s": c},
                    ensure_ascii=False,
                )
                + "\n"
            )


# --- GPU ---------------------------------------------------------------------------------------


def fast_cuda() -> None:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True  # safe: shapes are padded to whole seconds
    torch.set_float32_matmul_precision("high")


class GpuMonitor:
    """Samples nvidia-smi in a background thread; `window()` returns the mean utilisation and the
    peak memory since the previous call."""

    def __init__(self, every: float = 1.0):
        self.every, self._util, self._mem = every, [], []
        self._stop = threading.Event()
        self._thread = threading.Thread(target=self._run, daemon=True)

    def _run(self):
        while not self._stop.is_set():
            try:
                out = subprocess.run(
                    [
                        "nvidia-smi",
                        "--query-gpu=utilization.gpu,memory.used",
                        "--format=csv,noheader,nounits",
                    ],
                    capture_output=True,
                    text=True,
                    timeout=5,
                )
                util, mem = (float(x) for x in out.stdout.strip().splitlines()[0].split(","))
                self._util.append(util)
                self._mem.append(mem)
            except Exception:
                pass
            self._stop.wait(self.every)

    def start(self) -> GpuMonitor:
        self._thread.start()
        return self

    def stop(self) -> None:
        self._stop.set()

    def window(self) -> dict:
        util, mem = self._util, self._mem
        self._util, self._mem = [], []
        return {
            "gpu_util": float(np.mean(util)) if util else float("nan"),
            "gpu_mem_gib": max(mem) / 1024 if mem else float("nan"),
        }


def init_optimizer_state(model: torch.nn.Module, optimizer: torch.optim.Optimizer) -> None:
    """Allocate AdamW's moment buffers before probing, without moving any weight: with zero
    gradients and zero weight decay an AdamW step is a no-op, but it creates the state."""
    decay = [g["weight_decay"] for g in optimizer.param_groups]
    for g in optimizer.param_groups:
        g["weight_decay"] = 0.0
    for p in model.parameters():
        if p.requires_grad:
            p.grad = torch.zeros_like(p)
    optimizer.step()
    optimizer.zero_grad(set_to_none=True)
    for g, d in zip(optimizer.param_groups, decay, strict=True):
        g["weight_decay"] = d
    for state in optimizer.state.values():
        if "step" in state:
            state["step"].zero_()


def probe_max_items(
    step: Callable[[int], None], lo: int, hi: int, params: Sequence[torch.nn.Parameter]
) -> int:
    """The largest n in [lo, hi] for which `step(n)` (one forward+backward at the worst case)
    fits in memory. `step` must not touch the gradients.

    The gradient buffer stays allocated throughout, as it does in training from the second
    micro-batch of every accumulated step: freeing it between probes measured the activations
    without it and picked a batch that ran out of memory once training accumulated."""
    for p in params:
        p.grad = torch.zeros_like(p)
    best = 0
    while lo <= hi:
        mid = (lo + hi) // 2
        try:
            step(mid)
            torch.cuda.synchronize()
            best, lo = mid, mid + 1
        except torch.cuda.OutOfMemoryError:
            hi = mid - 1
        for p in params:
            p.grad.zero_()
        torch.cuda.empty_cache()
    for p in params:
        p.grad = None
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    return best


# --- training ----------------------------------------------------------------------------------


@dataclass
class TrainConfig:
    name: str
    out: str
    epochs: int = 10
    lr: float = 1e-5
    schedule: str = "linear"  # linear warmup+decay, or "tristage" (fairseq2's default)
    warmup_frac: float = 0.1
    effective_s: float = 720.0  # audio seconds per optimizer step, reached by accumulation
    weight_decay: float = 0.0
    clip_norm: float = 1.0
    patience: int = 3  # evaluations without a val-WER improvement before stopping
    seed: int = 0
    log_every: int = 10
    workers: int = 6


def lr_factor(step: int, total: int, cfg: TrainConfig) -> float:
    warm = max(1, int(cfg.warmup_frac * total))
    if step < warm:
        return (step + 1) / warm
    if cfg.schedule == "tristage":  # 10% warmup, 40% hold, 50% exponential decay to 5%
        hold_end = int(0.5 * total)
        if step < hold_end:
            return 1.0
        frac = (step - hold_end) / max(1, total - hold_end)
        return math.exp(math.log(0.05) * frac)
    return max(0.0, (total - step) / max(1, total - warm))


def group_steps(rows, batches, effective_s: float) -> list[list[list[int]]]:
    """Micro-batches grouped into optimizer steps of about `effective_s` seconds of real audio."""
    steps, cur, acc = [], [], 0.0
    for b in batches:
        cur.append(b)
        acc += sum(duration(rows[i]) for i in b)
        if acc >= effective_s:
            steps.append(cur)
            cur, acc = [], 0.0
    if cur:
        steps.append(cur)
    return steps


def retry_note(val: dict) -> str:
    """For an `evaluate` that wraps its decode in RetryLoops and reports the greedy score too."""
    if "retried" not in val:
        return ""
    return f"  (greedy WER {val['greedy_wer']:.2f}, {val['retried']} retried)"


def speed_check(
    model: torch.nn.Module,
    *,
    rows: Sequence[dict],
    batches: list[list[int]],
    collate: Callable[[list[dict]], Any],
    loss_fn: Callable[[torch.nn.Module, Any], tuple[torch.Tensor, int]],
    evaluate: Callable[[torch.nn.Module], dict],
    val_rows: Sequence[dict],
    gold_rows: Sequence[dict],
    epochs: int,
    monitor: GpuMonitor,
    workers: int = 6,
    n_micro: int = 10,
) -> dict:
    """Time the run before committing to it: forward+backward on `n_micro` real training
    micro-batches (no optimizer step, so no weight moves), then one full val pass, which is also
    the val WER before training. Projects the wall time of every epoch and the gold pass.

    The micro-batches run twice and only the second pass is timed, so cudnn's per-shape kernel
    search and worker start-up are excluded. Gradients stay allocated between micro-batches, as
    they do under accumulation, so the memory reading is training's. BatchNorm running
    statistics are restored after."""
    bns = [m for m in model.modules() if isinstance(m, torch.nn.modules.batchnorm._BatchNorm)]
    saved = [{k: v.clone() for k, v in m.state_dict().items()} for m in bns]
    sample = batches[:n_micro]
    it = iter(loader(rows, sample + sample, collate, workers))

    def run() -> tuple[float, float, int]:
        audio = padded = 0.0
        clips = 0
        for _ in sample:
            b = next(it)
            with torch.autocast("cuda", dtype=torch.bfloat16):
                loss, _ = loss_fn(model, b)
            loss.backward()
            model.zero_grad(set_to_none=False)
            audio, padded = audio + b["seconds"], padded + b["padded_seconds"]
            clips += len(b["lens"]) if "lens" in b else b["wav"].shape[0]
        torch.cuda.synchronize()
        return audio, padded, clips

    model.train()
    run()
    monitor.window()
    t0 = time.perf_counter()
    audio, padded, clips = run()
    train_dt = time.perf_counter() - t0
    gpu = monitor.window()
    model.zero_grad(set_to_none=True)
    for m, s in zip(bns, saved, strict=True):
        m.load_state_dict(s)

    model.eval()
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        val = evaluate(model)
    torch.cuda.synchronize()
    val_dt = time.perf_counter() - t0
    model.train()

    secs = lambda rs: sum(duration(r) for r in rs)  # noqa: E731
    epoch_s = secs(rows) / (audio / train_dt)
    gold_s = val_dt * secs(gold_rows) / secs(val_rows)
    rec = {
        "train_x_realtime": audio / train_dt,
        "train_ms_per_clip": 1000 * train_dt / clips,
        "padding_waste": 1 - audio / padded,
        **gpu,
        "val_s": val_dt,
        "val_ms_per_clip": 1000 * val_dt / len(val_rows),
        **{f"val_before_{k}": v for k, v in val.items()},
        "epoch_min": epoch_s / 60,
        "projected_h": (epochs * (epoch_s + val_dt) + gold_s) / 3600,
    }
    print(
        f"train: {rec['train_x_realtime']:.0f}x realtime = "
        f"{rec['train_ms_per_clip']:.0f} ms per clip "
        f"(forward+backward, {clips} clips), pad waste {rec['padding_waste']:.0%}, "
        f"GPU {rec['gpu_util']:.0f}% util, {rec['gpu_mem_gib']:.1f} GiB\n"
        f"val:   {val_dt:.0f} s for {len(val_rows)} clips = "
        f"{rec['val_ms_per_clip']:.0f} ms per clip "
        f"(batched decode); WER before training {val['wer']:.2f}, loops {val['loops']}"
        f"{retry_note(val)}\n"
        f"projected: {rec['epoch_min']:.1f} min per epoch + {val_dt / 60:.1f} min val -> "
        f"at most {rec['projected_h']:.2f} h for {epochs} epochs and gold "
        "(early stopping can cut it)",
        flush=True,
    )
    return rec


def train(
    model: torch.nn.Module,
    *,
    cfg: TrainConfig,
    rows: Sequence[dict],
    make_batches: Callable[[int], list[list[int]]],
    collate: Callable[[list[dict]], Any],
    loss_fn: Callable[[torch.nn.Module, Any], tuple[torch.Tensor, int]],
    evaluate: Callable[[torch.nn.Module], dict],
    save_best: Callable[[torch.nn.Module], None],
    optimizer: torch.optim.Optimizer,
    monitor: GpuMonitor,
) -> dict:
    """Train with bf16 autocast and gradient accumulation; evaluate on val after every epoch;
    keep the best weights (by val WER) in CPU memory and hand them to `save_best`.

    `loss_fn` returns a *summed* loss and the number of units it sums over (clips for CTC, tokens
    for cross-entropy); gradients are divided by the step's total units, so accumulated
    micro-batches of different sizes are weighted exactly. Count the units on the CPU (in
    `collate`): an `int()` of a GPU tensor stalls the host once per micro-batch."""
    out = Path(cfg.out)
    out.mkdir(parents=True, exist_ok=True)
    torch.manual_seed(cfg.seed)
    plan = [group_steps(rows, make_batches(e), cfg.effective_s) for e in range(cfg.epochs)]
    total = sum(len(p) for p in plan)
    base_lrs = [g["lr"] for g in optimizer.param_groups]
    params = [p for p in model.parameters() if p.requires_grad]
    history, best, best_state, bad, step = [], float("inf"), None, 0, 0
    print(
        f"{cfg.name}: {total} optimizer steps over {cfg.epochs} epochs "
        f"(~{cfg.effective_s / 60:.0f} min of audio each), peak lr {cfg.lr:g}, {cfg.schedule}"
    )
    for epoch in range(cfg.epochs):
        model.train()
        flat = [b for s in plan[epoch] for b in s]
        sizes = [len(s) for s in plan[epoch]]
        it = iter(loader(rows, flat, collate, cfg.workers))
        t_log, audio_log, padded_log, loss_log, units_log = time.perf_counter(), 0.0, 0.0, 0.0, 0
        monitor.window()
        for n_micro in sizes:
            for g, lr0 in zip(optimizer.param_groups, base_lrs, strict=True):
                g["lr"] = lr0 * lr_factor(step, total, cfg)
            step_units = 0
            for _ in range(n_micro):
                batch = next(it)
                with torch.autocast("cuda", dtype=torch.bfloat16):
                    loss, units = loss_fn(model, batch)
                loss.backward()
                step_units += units
                loss_log += loss.detach()  # stays on the GPU: no host sync per micro-batch
                units_log += units
                audio_log += batch["seconds"]
                padded_log += batch["padded_seconds"]
            torch._foreach_div_([p.grad for p in params if p.grad is not None], max(step_units, 1))
            gnorm = torch.nn.utils.clip_grad_norm_(params, cfg.clip_norm)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            step += 1
            if step % cfg.log_every == 0:
                torch.cuda.synchronize()
                dt = time.perf_counter() - t_log
                gpu = monitor.window()
                rec = {
                    "step": step,
                    "epoch": epoch + 1,
                    "lr": optimizer.param_groups[0]["lr"],
                    "loss": float(loss_log) / max(units_log, 1),
                    "grad_norm": float(gnorm),
                    "audio_x_realtime": audio_log / dt,
                    "padding_waste": 1 - audio_log / max(padded_log, 1e-9),
                    "peak_alloc_gib": torch.cuda.max_memory_allocated() / 2**30,
                    **gpu,
                }
                history.append(rec)
                print(
                    f"step {step:5d}/{total} ep {epoch + 1} lr {rec['lr']:.2e} "
                    f"loss {rec['loss']:.4f} | {rec['audio_x_realtime']:6.0f}x realtime, "
                    f"pad waste {rec['padding_waste']:.0%}, "
                    f"GPU {rec['gpu_util']:.0f}% util, {rec['gpu_mem_gib']:.1f} GiB",
                    flush=True,
                )
                t_log, audio_log, padded_log, loss_log, units_log = (
                    time.perf_counter(),
                    0.0,
                    0.0,
                    0.0,
                    0,
                )
        model.eval()
        with torch.no_grad():
            val = evaluate(model)
        history.append(
            {"epoch": epoch + 1, "step": step, **{f"val_{k}": v for k, v in val.items()}}
        )
        improved = val["wer"] < best
        print(
            f"== epoch {epoch + 1}: val WER {val['wer']:.2f}  CER {val['cer']:.2f}  raw WER "
            f"{val['raw_wer']:.2f}  loops {val['loops']}{retry_note(val)}"
            + ("  (best)" if improved else ""),
            flush=True,
        )
        (out / "history.json").write_text(json.dumps(history, indent=1))
        if improved:
            best, bad = val["wer"], 0
            best_state = {k: v.detach().to("cpu", copy=True) for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= cfg.patience:
                print(f"no val improvement in {cfg.patience} evaluations; stopping")
                break
    monitor.window()
    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    save_best(model)
    (out / "config.json").write_text(json.dumps(asdict(cfg), indent=1))
    return {"best_val_wer": best, "steps": step, "history": history}

In [ ]:
sys.path.insert(0, str(FT))
import warnings

import torch
import transformers
from huggingface_hub import snapshot_download

import ftkit

# transformers' generate() warns on every call (max_new_tokens vs max_length); it is noise here and
# buries the training log. Errors still show.
transformers.logging.set_verbosity_error()
warnings.filterwarnings("ignore", module="transformers")
ftkit.fast_cuda()
DATA = ftkit.download_dataset()
splits = ftkit.load_splits(DATA)
store = ftkit.AudioStore(DATA, [r["episode_id"] for rows in splits.values() for r in rows])
score = ftkit.harness_scorer(DATA, FT)
print({k: len(v) for k, v in splits.items()}, f"audio in RAM: {store.gib:.1f} GiB")

FLEX_DIR = snapshot_download(MODEL_ID)
sys.path.insert(0, FLEX_DIR)
from indic_transcribe import MODES, IndicTranscribe  # noqa: E402

asr = IndicTranscribe.from_pretrained(FLEX_DIR, device="cuda", dtype=torch.float32)
model, featurize, tk = asr.model, asr.fe, asr.tokenizer
itn, romanized = MODES[MODE]
PROMPT = tk.encode_prompt(LANG, itn=itn, romanized=romanized)
EOS, PAD, OFFSET = tk.eos_id, tk.pad_id, tk.spl_size
UNK_MULTI = tk.multi.unk_id()

DEV_DIGITS = str.maketrans("०१२३४५६७८९", "0123456789")


def normalize(text: str) -> str:
    """Map the characters the tokenizer lacks onto ones fold.py scores as identical."""
    return (text.replace("।", ".").translate(DEV_DIGITS).replace("—", "-")
            .replace("‍", "").replace("‌", ""))


for name in ("train", "val"):
    kept = []
    for r in splits[name]:
        ids = tk.multi.encode(normalize(r["text"]), out_type=int)
        if UNK_MULTI not in ids:
            r["ids"] = [OFFSET + i for i in ids] + [EOS]
            kept.append(r)
    print(f"{name}: dropped {len(splits[name]) - len(kept)} clip(s) the tokenizer cannot write")
    splits[name] = kept
r = splits["train"][0]
assert tk.decode(r["ids"][:-1]) == normalize(r["text"]).strip(), "target encoding does not round-trip"
print("prompt", PROMPT, "| longest target", max(len(r["ids"]) for r in splits["train"]), "tokens")

## Model, optimizer and the batch-size probe

In [ ]:
import torch.utils.checkpoint

if GRAD_CKPT:
    for layer in model.model.encoder.layers:
        layer._forward = layer.forward
        layer.forward = lambda *a, _l=layer, **k: torch.utils.checkpoint.checkpoint(_l._forward, *a, use_reentrant=False, **k)
model.train()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.0, fused=True)
ftkit.init_optimizer_state(model, optimizer)


def spec_augment(feats: torch.Tensor, lens: torch.Tensor) -> torch.Tensor:
    """Canary's defaults: 2 frequency masks of <= 27 bins, 10 time masks of <= 5% of each clip."""
    b, f, t = feats.shape
    fr = torch.arange(f, device=feats.device)[None, :]
    tr = torch.arange(t, device=feats.device)[None, :]
    mask = torch.zeros(b, f, t, dtype=torch.bool, device=feats.device)
    for _ in range(2):
        w = torch.randint(0, 28, (b, 1), device=feats.device)
        s = (torch.rand(b, 1, device=feats.device) * (f - w)).long()
        mask |= ((fr >= s) & (fr < s + w))[:, :, None]
    for _ in range(10):
        w = (torch.rand(b, 1, device=feats.device) * 0.05 * lens[:, None]).long()
        s = (torch.rand(b, 1, device=feats.device) * (lens[:, None] - w).clamp(min=1)).long()
        mask |= ((tr >= s) & (tr < s + w))[:, None, :]
    return feats.masked_fill(mask, 0.0)


def features(wav: torch.Tensor, lens: torch.Tensor):
    feats, flens = featurize(wav, lens)  # fp32 even inside autocast: it disables autocast itself
    mask = (torch.arange(feats.size(2), device=feats.device)[None, :] < flens[:, None]).long()
    return feats, flens, mask


def forward_loss(feats, mask, inp, lab):
    logits = model(input_features=feats, attention_mask=mask, decoder_input_ids=inp, use_cache=False).logits
    return torch.nn.functional.cross_entropy(logits.float().flatten(0, 1), lab.flatten(), ignore_index=-100,
                                             reduction="sum")


longest = max(splits["train"], key=ftkit.duration)
max_len = ftkit.pad_len(round(ftkit.duration(longest) * ftkit.SR), PAD_TO_S)
max_t = len(PROMPT) + max(len(r["ids"]) for r in splits["train"]) - 1


def probe_step(n):
    wav = torch.randn(n, max_len, device="cuda") * 0.1
    feats, _, mask = features(wav, torch.full((n,), max_len, device="cuda"))
    inp = torch.randint(OFFSET, OFFSET + 6000, (n, max_t), device="cuda")
    with torch.autocast("cuda", dtype=torch.bfloat16):
        loss = forward_loss(feats, mask, inp, inp)
    loss.backward()


max_items = ftkit.probe_max_items(probe_step, 1, 256, [p for p in model.parameters() if p.requires_grad])
BUDGET_S = max_items * max_len / ftkit.SR * PROBE_FRACTION
print(f"largest micro-batch at {max_len / ftkit.SR:.0f} s x {max_t} tokens: {max_items} clips -> "
      f"budget {BUDGET_S:.0f} s of padded audio per micro-batch")

## Batches, loss and decoding

In [ ]:
import math

N_PROMPT = len(PROMPT)


def collate(rows):
    clips = [torch.from_numpy(store.clip(r).copy()) for r in rows]
    wav = torch.zeros(len(rows), ftkit.pad_len(max(len(c) for c in clips), PAD_TO_S), dtype=torch.int16)
    for i, c in enumerate(clips):
        wav[i, :len(c)] = c
    width = N_PROMPT + max(len(r["ids"]) for r in rows) - 1
    inp = torch.full((len(rows), width), PAD, dtype=torch.long)
    lab = torch.full((len(rows), width), -100, dtype=torch.long)
    for i, r in enumerate(rows):
        full = PROMPT + r["ids"]
        inp[i, :len(full) - 1] = torch.tensor(full[:-1])
        lab[i, N_PROMPT - 1:len(full) - 1] = torch.tensor(r["ids"])  # predict target + eos only
    return {"wav": wav, "lens": torch.tensor([len(c) for c in clips]), "inp": inp, "lab": lab,
            "tokens": sum(len(r["ids"]) for r in rows),
            "seconds": sum(ftkit.duration(r) for r in rows), "padded_seconds": wav.numel() / ftkit.SR}


def loss_fn(model, b):
    wav = b["wav"].cuda(non_blocking=True).float() / 32768.0
    feats, flens, mask = features(wav, b["lens"].cuda(non_blocking=True))
    feats = spec_augment(feats, flens)
    loss = forward_loss(feats, mask, b["inp"].cuda(non_blocking=True), b["lab"].cuda(non_blocking=True))
    return loss, b["tokens"]


def transcribe(rows, **gen):
    clips = [store.clip_f32(r) for r in rows]
    wav = torch.zeros(len(rows), ftkit.pad_len(max(len(c) for c in clips), PAD_TO_S))
    for i, c in enumerate(clips):
        wav[i, :len(c)] = c
    feats, _, mask = features(wav.cuda(), torch.tensor([len(c) for c in clips], device="cuda"))
    prompt = torch.tensor([PROMPT] * len(rows), device="cuda")
    with torch.autocast("cuda", dtype=torch.bfloat16):
        out = model.generate(input_features=feats, attention_mask=mask, decoder_input_ids=prompt,
                             do_sample=False, num_beams=1, eos_token_id=EOS, pad_token_id=PAD,
                             **{"max_new_tokens": 300, **gen})
    return [tk.decode(tk.strip_prompt_and_trim(o.tolist(), PROMPT)) for o in out]


# Loop retry, fixed before scoring anything with it: greedy decoding occasionally sticks on a filler
# ("अँ. अँ. अँ.") to the length limit. A global repetition ban would also hit real repeats the
# references keep ("कोही छैन। कोही छैन"), so only a clip whose greedy output loops is decoded
# again, alone, with a repetition penalty, no repeated 6-token phrase, and a length cap from its
# duration (the densest training label is 12.6 tokens/s).
MAX_TOKENS_PER_S = 13.0
RETRY = {"repetition_penalty": 1.2, "no_repeat_ngram_size": 6}


def retry_one(row):
    cap = min(300, math.ceil(MAX_TOKENS_PER_S * ftkit.duration(row)))
    return transcribe([row], max_new_tokens=cap, **RETRY)[0]


def evaluate(model, rows=None):
    rows = rows or splits["val"]
    texts, _ = ftkit.transcribe_rows(rows, ftkit.RetryLoops(transcribe, retry_one), budget_s=EVAL_BUDGET_S,
                                     max_items=EVAL_ITEMS, pad_to_s=PAD_TO_S)
    return score([r["text"] for r in rows], texts)


def save_best(model):
    import shutil

    dst = OUT / "best"
    model.save_pretrained(dst, state_dict={k: v.to(torch.bfloat16) for k, v in model.state_dict().items()})
    for f in Path(FLEX_DIR).iterdir():  # code, tokenizer and feature files, so IndicTranscribe loads it
        if f.suffix in {".py", ".model", ".md"} or f.name in {"tokenizer_config.json", "generation_config.json",
                                                             "feature_extractor.safetensors"}:
            shutil.copy(f, dst / f.name)


def make_batches(epoch):
    return ftkit.bucket_batches(splits["train"], budget_s=BUDGET_S, max_items=256, pad_to_s=PAD_TO_S,
                                shuffle=True, seed=epoch)


monitor = ftkit.GpuMonitor().start()

## One clip per call vs batched

The zero-shot comparisons decoded one clip per call (`asr(path)`), at about 1.4 s per clip
(RTF 0.12). Here the same 16 val clips are decoded both ways, under the same bf16 autocast, so the
only difference is batching. The cell reports the speed-up and whether the texts agree; the port's
batched `generate` had never been run before.

In [ ]:
import time

sample = splits["val"][:16]
model.eval()
with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
    transcribe(sample)  # warm-up
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    single = [asr.transcribe(store.clip_f32(r), lang=LANG, mode=MODE, max_new_tokens=300) for r in sample]
    torch.cuda.synchronize()
    t1 = time.perf_counter()
    batched = transcribe(sample)
    torch.cuda.synchronize()
    t2 = time.perf_counter()
model.train()
same = sum(a.strip() == b.strip() for a, b in zip(single, batched))
print(f"one clip per call: {(t1 - t0) / len(sample):.2f} s per clip | batched: {1000 * (t2 - t1) / len(sample):.0f} ms "
      f"per clip ({(t1 - t0) / (t2 - t1):.0f}x) | identical text on {same}/{len(sample)} clips, folded WER of "
      f"batched against one-per-call {score(single, batched)['wer']:.2f}%")

## Speed check (before committing to the run)

This times forward+backward on 10 real training micro-batches, then one full batched val pass,
and projects the whole run. The val pass doubles as the **val WER before training**, the baseline
that fine-tuning has to beat. No optimizer step runs, so no weight moves. If the projection is
too long, or GPU utilisation is low, stop here and fix it.

In [ ]:
speed = ftkit.speed_check(model, rows=splits["train"], batches=make_batches(0), collate=collate,
                          loss_fn=loss_fn, evaluate=evaluate, val_rows=splits["val"], gold_rows=splits["gold"],
                          epochs=EPOCHS, monitor=monitor)
(OUT / "speed_check.json").write_text(json.dumps(speed, indent=1))

## Train

In [ ]:
cfg = ftkit.TrainConfig(name=RUN_NAME, out=str(OUT), epochs=EPOCHS, lr=LR, warmup_frac=WARMUP,
                        effective_s=EFFECTIVE_S, patience=2)
result = ftkit.train(model, cfg=cfg, rows=splits["train"], make_batches=make_batches, collate=collate,
                     loss_fn=loss_fn, evaluate=evaluate, save_best=save_best, optimizer=optimizer,
                     monitor=monitor)
print("best val WER", result["best_val_wer"])

## Gold
The gold hypotheses go to `OUT/hyps/<RUN_NAME>.jsonl`, in the bake-off's cache format. Copy them to
the bake-off checkout's `hyps/` folder to score them beside the zero-shot systems.

In [ ]:
decode = ftkit.RetryLoops(transcribe, retry_one)
texts, compute = ftkit.transcribe_rows(splits["gold"], decode, budget_s=EVAL_BUDGET_S,
                                       max_items=EVAL_ITEMS, pad_to_s=PAD_TO_S)
refs = [r["text"] for r in splits["gold"]]
gold = score(refs, texts)
gold["rtf"] = sum(compute) / sum(ftkit.duration(r) for r in splits["gold"])
first = {sid: text for sid, text, _ in decode.log}  # the same run's greedy output, before any retry
gold["greedy_only"] = score(refs, [first.get(r["segment_id"], t) for r, t in zip(splits["gold"], texts)])
gold["retried"] = [{"segment_id": s, "first": f, "retry": t} for s, f, t in decode.log]
(OUT / "gold_metrics.json").write_text(json.dumps(gold, indent=1, ensure_ascii=False))
ftkit.write_hyps(OUT / "hyps" / f"{RUN_NAME}.jsonl", splits["gold"], texts, compute)
print("with loop retry:", {k: v for k, v in gold.items() if k not in ("greedy_only", "retried")})
print("greedy only:    ", gold["greedy_only"])
for s, f, t in decode.log:
    print(f"\n{s}\n  greedy: ...{f[-90:]}\n  retry:  ...{t[-90:]}")

## Harness model folder

Everything the harness's **Models** page needs to show this run (D83): the model card, plus gold
and val transcripts from the best weights under the same decoder. No weights: the page scores
text, it never runs the model. Copy `OUT/harness/` from Drive to the harness checkout as
`data/models/asr/<slug>/` (the folder name becomes the model's id), then press **Rescan** on the
Models page. The harness scores the text against its *current* labels, so a clip relabeled since
this export is scored against the new label.

In [ ]:
import datetime as dt

HARNESS = OUT / "harness"
val_decode = ftkit.RetryLoops(transcribe, retry_one)
val_texts, val_compute = ftkit.transcribe_rows(splits["val"], val_decode, budget_s=EVAL_BUDGET_S,
                                               max_items=EVAL_ITEMS, pad_to_s=PAD_TO_S)
ftkit.write_hyps(HARNESS / "gold.jsonl", splits["gold"], texts, compute)
ftkit.write_hyps(HARNESS / "val.jsonl", splits["val"], val_texts, val_compute)
evals = [h for h in result["history"] if "val_wer" in h]
export = json.loads((DATA / "training" / "manifest.json").read_text())
card = {
    "name": MODEL_NAME,
    "created_at": dt.datetime.now(dt.UTC).isoformat(timespec="seconds"),
    "description": MODEL_DESCRIPTION,
    "architecture": "Canary-style enc-dec: 32L conformer + 24L transformer decoder, 1.2B",
    "base_model": MODEL_ID,
    "decoder": "greedy+retry",
    "run_name": RUN_NAME,
    "epochs": EPOCHS,
    "best_epoch": min(evals, key=lambda h: h["val_wer"])["epoch"] if evals else None,
    "lr": LR,
    "val_wer": result["best_val_wer"],
    "gold_wer": gold["wer"],
    "train_export": {k: export.get(k) for k in ("exported_at", "git_commit", "row_count",
                                                "normalization_version", "label_version")},
}
(HARNESS / "model_card.json").write_text(json.dumps(card, indent=1, ensure_ascii=False))
print(f"val {score([r['text'] for r in splits['val']], val_texts)['wer']:.2f}% | wrote", HARNESS)

## Results

In [ ]:
import json

import matplotlib.pyplot as plt

hist = json.loads((OUT / "history.json").read_text())
steps = [h for h in hist if "loss" in h]
evals = [h for h in hist if "val_wer" in h]
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
axes[0].plot([h["step"] for h in steps], [h["loss"] for h in steps], color="#2a78d6")
axes[0].set_title("train loss per unit", loc="left")
axes[1].plot([h["epoch"] for h in evals], [h["val_wer"] for h in evals], marker="o", color="#2a78d6")
axes[1].set_title("val folded WER %", loc="left")
axes[2].plot([h["step"] for h in steps], [h["gpu_util"] for h in steps], color="#2a78d6")
axes[2].set_ylim(0, 100)
axes[2].set_title("GPU utilisation %", loc="left")
for ax in axes:
    ax.grid(color="#ecebe7")
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()
print(json.dumps(json.loads((OUT / "gold_metrics.json").read_text()), indent=1))

## CPU export: quantize, benchmark, export

The target is a CPU at a desk, not this GPU: the harness's mic playground (D85).
**What was measured locally (Ryzen 7 7700X, 8 threads, 2026-09-15)**, base Flex on 10 gold clips:
- fp32 runs at RTF 0.41 and bf16 at RTF 0.18, with the same WER (21.8%). bf16 is realtime and
  free, and `best/` already stores bf16, so it is the CPU model if nothing else passes.
- PyTorch's **dynamic int8** (activations quantized too) broke the model: WER 97-113%, loops, and
  English written in Devanagari. It is not tried here.
- **Weight-only int8** (`cpukit.py`: int8 weights with one scale per output channel, bf16
  activations) decodes 1.5x faster than bf16 on a full-size Flex (16.5 vs 24.5 ms/token, random
  weights). Decoding is memory-bound, so halving the weight bytes is where CPU speed comes from.

**What this section measures** is the part that was not measured locally: what int8 costs in
accuracy on *this* model.
1. **Accuracy, on the GPU.** The trained model is quantized in place (this is its last use) and
   val and gold are decoded with the same decoder as above. The int8 arithmetic is dequantized on
   the GPU, so the numbers carry over to the CPU kernel.
2. **Rule, fixed before any run:** export int8 if its val WER is within **+0.3** of bf16 (the
   run-to-run noise) and it loops no more often. Otherwise the CPU model is `best/` in bf16.
3. **Speed, on this VM's CPU.** The same `CPU_TIMING_CLIPS` val clips are timed for each variant.
   Colab's CPU is not your machine (it may lack bf16 instructions entirely), so only the ratio
   between variants means anything. Measure the absolute speed where the model will run.
4. **Export.** An accepted int8 model goes to `OUT/cpu/` (~1.3 GB against 2.5 GB, with its code
   and `cpukit.py`; load it with `cpukit.load_int8`). `cpu_bench.json` holds every number, and the
   harness model card gets a `cpu` block.

ONNX is not needed: plain PyTorch bf16 is already realtime on the target CPU.

In [ ]:
%%writefile /content/ft/cpukit.py
"""Weight-only int8 for Indic-Transcribe-Flex on a CPU (04c's CPU export).

Written out by a %%writefile cell in 04c and copied into `OUT/cpu/`, so the export loads anywhere
with plain PyTorch -- no torchao, no ONNX.

**Why weight-only.** Measured on the owner's Ryzen 7 7700X, 2026-09-15, base Flex on 10 gold clips:
fp32 RTF 0.41, bf16 RTF 0.18 at the same WER (21.8%). PyTorch's *dynamic* int8, which also
quantizes activations, broke the model: WER 97-113%, loops, and English written in Devanagari.
Decoding is memory-bound (each token re-reads the 24 decoder layers), so the speed lives in the
weight bytes. This keeps activations in bf16 and stores weights as int8 with one scale per output
channel: half the bytes of bf16 per token, none of dynamic int8's activation rounding.

Only the transformer layers are quantized. `pre_encode.out` must keep real parameters (the encoder
reads its dtype from them) and the tied `lm_head` is small.
"""

from __future__ import annotations

import json
import math
import time
from pathlib import Path

import torch
from torch import nn

QUANT_MANIFEST = "int8_modules.json"
INT8_WEIGHTS = "model.int8.safetensors"


class Int8Linear(nn.Module):
    """y = x @ (q * s)^T + b with q int8 (out, in) and one scale s per output channel."""

    def __init__(self, qweight: torch.Tensor, scales: torch.Tensor, bias: torch.Tensor | None):
        super().__init__()
        self.out_features, self.in_features = qweight.shape
        self.register_buffer("qweight", qweight)
        self.register_buffer("scales", scales)
        self.register_buffer("bias", bias)

    @classmethod
    def from_linear(cls, linear: nn.Linear, dtype: torch.dtype) -> Int8Linear:
        # Quantize from the bf16 values `best/` stores, so the fp32 model scored on the GPU and the
        # CPU export built from `best/` hold exactly the same int8 weights.
        w = linear.weight.detach().to(torch.bfloat16).float()
        scales = w.abs().amax(dim=1).clamp(min=1e-8) / 127.0
        q = torch.round(w / scales[:, None]).clamp(-127, 127).to(torch.int8)
        bias = None if linear.bias is None else linear.bias.detach().to(dtype)
        return cls(q, scales.to(dtype), bias)

    @classmethod
    def empty(
        cls, in_features: int, out_features: int, bias: bool, dtype: torch.dtype
    ) -> Int8Linear:
        return cls(
            torch.zeros(out_features, in_features, dtype=torch.int8),
            torch.ones(out_features, dtype=dtype),
            torch.zeros(out_features, dtype=dtype) if bias else None,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        shape = x.shape
        x2 = x.reshape(-1, self.in_features)
        if x2.is_cuda or not hasattr(torch, "_weight_int8pack_mm"):
            # The same arithmetic, dequantized: on the GPU (accuracy only) or on an older torch.
            y = x2 @ (self.qweight.to(x2.dtype) * self.scales.to(x2.dtype)[:, None]).t()
        else:
            # PyTorch's CPU kernel for int8 weights with bf16/fp32 activations.
            y = torch._weight_int8pack_mm(x2.contiguous(), self.qweight, self.scales.to(x2.dtype))
        if self.bias is not None:
            y = y + self.bias.to(y.dtype)
        return y.reshape(*shape[:-1], self.out_features)


def _targets(model: nn.Module) -> list[str]:
    """Names of the nn.Linear modules inside the encoder and decoder layers."""
    inner = model.model
    names = []
    for part in ("encoder", "decoder"):
        for name, module in getattr(inner, part).layers.named_modules():
            if isinstance(module, nn.Linear):
                names.append(f"model.{part}.layers.{name}")
    return names


def _set(model: nn.Module, name: str, module: nn.Module) -> None:
    parent, _, leaf = name.rpartition(".")
    setattr(model.get_submodule(parent), leaf, module)


def quantize_(model: nn.Module, dtype: torch.dtype = torch.bfloat16) -> list[str]:
    """Swap every transformer-layer Linear for an Int8Linear, in place; returns their names."""
    names = _targets(model)
    for name in names:
        _set(model, name, Int8Linear.from_linear(model.get_submodule(name), dtype))
    return names


def save_int8(model: nn.Module, names: list[str], src: Path, dst: Path) -> None:
    """Write a loadable CPU folder: int8 weights, the list of quantized modules, and the model's
    code, tokenizer and feature files copied from ``src`` (a `best/` folder)."""
    import shutil

    from safetensors.torch import save_file

    dst.mkdir(parents=True, exist_ok=True)
    state = {k: v.detach().cpu().contiguous() for k, v in model.state_dict().items()}
    # lm_head is tied to the decoder embedding; keep one copy, as save_pretrained does.
    state.pop("lm_head.weight", None)
    save_file(state, str(dst / INT8_WEIGHTS))
    (dst / QUANT_MANIFEST).write_text(json.dumps(names))
    for f in src.iterdir():
        if f.name != "model.safetensors" and f.is_file():
            shutil.copy(f, dst / f.name)
    shutil.copy(Path(__file__), dst / "cpukit.py")


def load_int8(path: Path | str, threads: int | None = None):
    """An IndicTranscribe on the CPU from a folder written by :func:`save_int8`.

    Usage, from anywhere::

        import sys; sys.path.insert(0, "<folder>")
        import cpukit
        asr = cpukit.load_int8("<folder>")
        asr.transcribe("clip.wav", lang="ne", mode="mixed")
    """
    import sys

    from safetensors.torch import load_file

    path = Path(path)
    sys.path.insert(0, str(path))
    from configuration_indic_canary import IndicCanaryConfig
    from feature_extraction_indic_canary import IndicCanaryFeatureExtractor
    from indic_transcribe import IndicTranscribe
    from modeling_indic_canary import IndicCanaryForConditionalGeneration
    from tokenization_indic_canary import IndicCanaryTokenizer

    if threads:
        torch.set_num_threads(threads)
    config = IndicCanaryConfig.from_pretrained(path)
    previous = torch.get_default_dtype()
    torch.set_default_dtype(torch.bfloat16)  # build the skeleton at half the RAM of fp32
    try:
        model = IndicCanaryForConditionalGeneration(config)
    finally:
        torch.set_default_dtype(previous)
    for name in json.loads((path / QUANT_MANIFEST).read_text()):
        linear = model.get_submodule(name)
        _set(
            model,
            name,
            Int8Linear.empty(
                linear.in_features, linear.out_features, linear.bias is not None, torch.bfloat16
            ),
        )
    missing, unexpected = model.load_state_dict(load_file(str(path / INT8_WEIGHTS)), strict=False)
    missing = [k for k in missing if k != "lm_head.weight"]
    if missing or unexpected:
        raise RuntimeError(
            f"int8 export does not fit the model: missing {missing[:3]}, "
            f"unexpected {unexpected[:3]}"
        )
    model.tie_weights()
    model.eval()
    return IndicTranscribe(
        model,
        IndicCanaryFeatureExtractor.from_pretrained(path, device="cpu"),
        IndicCanaryTokenizer.from_pretrained(path),
        "cpu",
    )


def time_clips(
    asr, clips: list[tuple[str, torch.Tensor]], *, lang: str, mode: str, tokens_per_s: float = 13.0
) -> dict:
    """Greedy-decode each (id, 16 kHz float waveform) on the CPU: RTF, and wall time per decoded
    token (encoder included, so it slightly overstates the decoder's own cost).

    The first clip is decoded twice and only the second is timed (kernel selection, allocations).
    """
    from indic_transcribe import MODES

    itn, romanized = MODES[mode]
    prompt = asr.tokenizer.encode_prompt(lang, itn=itn, romanized=romanized)
    dtype = next(p for p in asr.model.parameters()).dtype
    texts, audio_s, total_s, tokens = {}, 0.0, 0.0, 0

    def one(wav):
        feats, mask = asr._features(wav)
        seconds = wav.shape[0] / asr.fe.sample_rate
        cap = min(300, math.ceil(tokens_per_s * seconds))
        with torch.inference_mode():
            t0 = time.perf_counter()
            out = asr.model.generate(
                input_features=feats.to(dtype),
                attention_mask=mask,
                decoder_input_ids=torch.tensor([prompt]),
                max_new_tokens=cap,
            )
            took = time.perf_counter() - t0
        text = asr.tokenizer.decode(asr.tokenizer.strip_prompt_and_trim(out[0].tolist(), prompt))
        return text, seconds, took, out.shape[1] - len(prompt)

    one(clips[0][1])
    for clip_id, wav in clips:
        text, seconds, took, n = one(wav)
        texts[clip_id] = text
        audio_s, total_s, tokens = audio_s + seconds, total_s + took, tokens + n
    return {
        "clips": len(clips),
        "audio_s": round(audio_s, 1),
        "rtf": round(total_s / audio_s, 3),
        "ms_per_token": round(1000 * total_s / max(1, tokens), 1),
        "tokens_per_audio_s": round(tokens / audio_s, 2),
        "texts": texts,
    }

In [ ]:
import cpukit

CPU_TIMING_CLIPS = 8   # val clips timed on this VM's CPU; accuracy uses all of val and gold on the GPU
MAX_WER_COST = 0.3     # points of val WER int8 may cost against bf16 (the run-to-run noise)

val_refs, gold_refs = [r["text"] for r in splits["val"]], [r["text"] for r in splits["gold"]]
bf16_scores = {"val": score(val_refs, val_texts), "gold": score(gold_refs, texts)}
cpukit.quantize_(model, dtype=torch.float32)  # in place: the GPU copy is not needed after this
int8_texts = {}
for name, rows in (("val", splits["val"]), ("gold", splits["gold"])):
    int8_texts[name], _ = ftkit.transcribe_rows(rows, ftkit.RetryLoops(transcribe, retry_one),
                                                budget_s=EVAL_BUDGET_S, max_items=EVAL_ITEMS, pad_to_s=PAD_TO_S)
int8_scores = {"val": score(val_refs, int8_texts["val"]), "gold": score(gold_refs, int8_texts["gold"])}
same = sum(a == b for a, b in zip(val_texts, int8_texts["val"]))
accept = (int8_scores["val"]["wer"] <= bf16_scores["val"]["wer"] + MAX_WER_COST
          and int8_scores["val"]["loops"] <= bf16_scores["val"]["loops"])
for name in ("val", "gold"):
    print(f"{name}: bf16 {bf16_scores[name]['wer']:.2f}% ({bf16_scores[name]['loops']} loops) | "
          f"int8 {int8_scores[name]['wer']:.2f}% ({int8_scores[name]['loops']} loops)")
print(f"int8 text identical to bf16 on {same}/{len(val_texts)} val clips ->",
      "EXPORT int8" if accept else "int8 rejected; the CPU model is best/ in bf16")

In [ ]:
import subprocess

step = max(1, len(splits["val"]) // CPU_TIMING_CLIPS)
timing_rows = sorted(splits["val"], key=ftkit.duration)[::step][:CPU_TIMING_CLIPS]
clips = [(r["segment_id"], torch.as_tensor(store.clip_f32(r)).float().cpu()) for r in timing_rows]
threads = os.cpu_count()
torch.set_num_threads(threads)
cpu_name = subprocess.run("lscpu | sed -n 's/^Model name: *//p'", shell=True, capture_output=True,
                          text=True).stdout.strip()
cpu_flags = open("/proc/cpuinfo").read()
host = {"cpu": cpu_name, "threads": threads, "avx512_bf16": "avx512_bf16" in cpu_flags,
        "amx_bf16": "amx_bf16" in cpu_flags}
print(host)

timing = {}
cpu_asr = IndicTranscribe.from_pretrained(str(OUT / "best"), device="cpu", dtype=torch.bfloat16)
timing["bf16"] = cpukit.time_clips(cpu_asr, clips, lang=LANG, mode=MODE)
if accept:
    names = cpukit.quantize_(cpu_asr.model)
    cpukit.save_int8(cpu_asr.model, names, OUT / "best", OUT / "cpu")
    del cpu_asr
    cpu_asr = cpukit.load_int8(OUT / "cpu")  # time what was written, not what is in memory
    timing["int8"] = cpukit.time_clips(cpu_asr, clips, lang=LANG, mode=MODE)
del cpu_asr
for variant, t in timing.items():
    print(f"{variant}: RTF {t['rtf']:.3f}, {t['ms_per_token']:.1f} ms/token over {t['audio_s']} s of audio")

bench = {
    "variant": "int8-weight-only" if accept else "bf16",
    "export": "cpu/" if accept else "best/",
    "rule": f"int8 if val WER <= bf16 + {MAX_WER_COST} and no more loops",
    "scores": {"bf16": bf16_scores, "int8": int8_scores},
    "int8_identical_val_texts": same,
    "timing_host": host,
    "timing": {v: {k: x for k, x in t.items() if k != "texts"} for v, t in timing.items()},
}
(OUT / "cpu_bench.json").write_text(json.dumps(bench, indent=1))
card = json.loads((HARNESS / "model_card.json").read_text())
card["cpu"] = {
    "variant": bench["variant"],
    "export": bench["export"],
    "val_wer_bf16": bf16_scores["val"]["wer"],
    "val_wer_int8": int8_scores["val"]["wer"],
    "timing_host": host["cpu"],
    "rtf": {v: t["rtf"] for v, t in timing.items()},
}
(HARNESS / "model_card.json").write_text(json.dumps(card, indent=1, ensure_ascii=False))
print("wrote", OUT / "cpu_bench.json", "and the card's cpu block")

## Playground bundle

The harness's Models page can run this model on the CPU and transcribe your voice (D85). It needs
the harness folder above plus the CPU weights this bench accepted, both in
`data/models/asr/<RUN_NAME>/`. This packs them into one uncompressed tar on Drive (safetensors do
not compress). Download it and unpack it under `data/models/asr/`, press **Rescan**, and start the
sidecar: `docker-compose --profile playground up -d playground`.

In [ ]:
import tarfile

export = json.loads((HARNESS / "model_card.json").read_text())["cpu"]["export"].strip("/")
bundle = OUT / f"{RUN_NAME}-playground.tar"
with tarfile.open(bundle, "w") as tar:
    for f in sorted(HARNESS.iterdir()):
        tar.add(f, arcname=f"{RUN_NAME}/{f.name}")
    tar.add(OUT / export, arcname=f"{RUN_NAME}/{export}")
print(bundle, f"{bundle.stat().st_size / 2**30:.2f} GiB,", export)